# Day 1: From messy data to clean insights
**Python for Data Science in Academic Research, with AI, VS Code & GitHub**
Nasarawa State University, Keffi · Friday 30 October 2026 · 10:00 am – 2:00 pm

### By the end of today you will be able to
1. Run Python notebooks inside **VS Code** in a project's own environment
2. Keep your research under **version control** with Git and GitHub
3. **Audit** a dataset for quality problems and keep a cleaning log for your methods chapter
4. **Clean** data with pandas and turn the steps into a reusable function
5. **Explore** data visually and use an **AI assistant** *responsibly* along the way

### How to read this notebook
| Symbol | Meaning |
|---|---|
| 🤖 | **AI exercise**: use ChatGPT, Claude, Gemini or GitHub Copilot, then **check** what it gives you |
| ✍️ | **Your turn**: write your own code or notes |
| 💾 | **Git checkpoint**: commit your work |

Run a cell with **Shift + Enter**. If VS Code asks for a kernel, choose the `.venv` environment you created in `docs/SETUP.md`.

## 0. Check your environment

In [ ]:
import platform, importlib

print("Python", platform.python_version())
for pkg in ["pandas", "numpy", "matplotlib", "seaborn", "sklearn", "statsmodels", "scipy", "streamlit"]:
    try:
        print(f"  ✅ {pkg:12s} {importlib.import_module(pkg).__version__}")
    except ImportError:
        print(f"  ❌ {pkg:12s} NOT INSTALLED - run: pip install -r requirements.txt")

## 1. A research project is a folder, not a single file

```
nsuk-ds-workshop-starter/
├── data/raw/         ← original data, NEVER edited by hand
├── data/processed/   ← created by code (can always be re-created)
├── notebooks/        ← exploration and analysis, in order (01_, 02_ ...)
├── src/              ← reusable functions (cleaning, modelling, plotting)
├── figures/          ← figures for your thesis or paper (created by code)
├── results/          ← tables and outputs (created by code)
├── models/           ← saved trained models
├── app/              ← Streamlit app (Day 2)
├── prompts/          ← your AI prompt library
└── docs/             ← setup guide, Git cheat sheet, checklists
```

**Golden rule:** anything in `processed/`, `figures/`, `results/` and `models/` should be re-creatable by running your code on `raw/`. That is what *reproducible* means.

### 💾 Git checkpoint 1
Open the VS Code terminal (**Ctrl + `**) and run:
```bash
git status
git add .
git commit -m "Start Day 1 notebook"
git push
```
Or use the **Source Control** panel (Ctrl + Shift + G). See `docs/GIT_CHEATSHEET.md`.

## 2. Load the raw data

> ⚠️ **About this dataset.** `nasarawa_maize_survey_synthetic.csv` is **simulated** data that imitates a smallholder maize survey across the 13 LGAs of Nasarawa State. It was made for teaching and contains **deliberate errors**. Never cite it as real evidence.

| Column | Meaning |
|---|---|
| `farm_id`, `survey_year`, `lga` | identifiers and location |
| `farmer_age`, `gender`, `education_level`, `years_farming` | farmer characteristics |
| `farm_size_ha`, `improved_seed`, `fertilizer_kg_per_ha`, `irrigation` | farm practices |
| `rainfall_mm`, `avg_temp_c`, `soil_ph` | environment |
| `extension_visits`, `cooperative_member`, `pest_incidence` | support and pests |
| `harvest_sales_naira` | money earned from the harvest |
| **`yield_kg_per_ha`** | **outcome we want to explain and predict** |

In [ ]:
import sys
from pathlib import Path

# Make the project's src/ folder importable, whether the notebook runs from
# the repository root or from inside notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.plotting import set_publication_style, save_figure

set_publication_style()
pd.set_option("display.max_columns", 30)
print("Project root:", ROOT)

In [ ]:
RAW_PATH = ROOT / "data" / "raw" / "nasarawa_maize_survey_synthetic.csv"
df_raw = pd.read_csv(RAW_PATH)

print(df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include='all').T

### 🤖 AI exercise 1: a second pair of eyes
Copy the output of `df_raw.info()` and `describe()` (**not the raw data itself**) into your AI assistant with this prompt:

> *"Act as a careful data science supervisor. Below is the `info()` and `describe()` output of a farm survey dataset. List the data-quality problems I should check before any analysis, and say which output line points to each one."*

Then **check each claim with code** in the cells below. AI can be confidently wrong. Your thesis is your responsibility.

🔒 **Never paste confidential, personal, or unpublished data into a public AI tool.** Summaries like `info()` are usually safe. Raw participant records are not.

## 3. Data quality audit
### 3.1 Missing values

In [ ]:
missing = pd.DataFrame({
    "n_missing": df_raw.isna().sum(),
    "pct_missing": (df_raw.isna().mean() * 100).round(1),
}).query("n_missing > 0").sort_values("pct_missing", ascending=False)
missing

### 3.2 Duplicates

In [ ]:
print("Fully duplicated rows:", df_raw.duplicated().sum())
print("Repeated farm IDs:    ", df_raw["farm_id"].duplicated().sum())

### 3.3 Inconsistent categories
There are only **13** LGAs in Nasarawa State. How many does the data claim?

In [ ]:
print(df_raw["lga"].nunique(), "distinct LGA values")
df_raw["lga"].value_counts()

In [ ]:
for col in ["improved_seed", "irrigation", "cooperative_member"]:
    print(col, df_raw[col].unique())

### 3.4 Numbers stored as text
`fertilizer_kg_per_ha` should be numeric, but `info()` says `object`. Why?

In [ ]:
fert_as_number = pd.to_numeric(df_raw["fertilizer_kg_per_ha"], errors="coerce")
df_raw.loc[fert_as_number.isna() & df_raw["fertilizer_kg_per_ha"].notna(), "fertilizer_kg_per_ha"].unique()

### 3.5 Impossible values

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, col in zip(axes, ["farmer_age", "farm_size_ha", "soil_ph"]):
    sns.boxplot(x=df_raw[col], ax=ax, color="#56B4E9")
    ax.set_title(col)
plt.tight_layout()

print("Ages outside 15-90:     ", (~df_raw["farmer_age"].between(15, 90) & df_raw["farmer_age"].notna()).sum())
print("Farms larger than 50 ha:", (df_raw["farm_size_ha"] > 50).sum())

### ✍️ Your turn: write a data cleaning log
Examiners will ask *"what did you do to the data?"*. Write your answer **now**, while you remember. Edit this cell:

| # | Problem found | How many rows | Decision | Reason |
|---|---|---|---|---|
| 1 | Duplicate rows | ? | Drop | Same farm entered twice |
| 2 | LGA spelling variants | ? | Map to official names | ... |
| 3 | ... | | | |

## 4. Clean the data, one step at a time

In [ ]:
df = df_raw.copy()          # never change df_raw, so you can always compare

before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} duplicate rows")

In [ ]:
lga_fixes = {"Nasarawa-Eggon": "Nasarawa Eggon", "N/Eggon": "Nasarawa Eggon", "Nassarawa": "Nasarawa"}
df["lga"] = df["lga"].str.strip().str.title().replace(lga_fixes)

print(df["lga"].nunique(), "LGAs now")
sorted(df["lga"].unique())

In [ ]:
yes_no = {"yes": "Yes", "y": "Yes", "no": "No", "n": "No"}
for col in ["improved_seed", "irrigation", "cooperative_member"]:
    df[col] = df[col].astype("string").str.strip().str.lower().map(yes_no)
    print(col, df[col].unique())

In [ ]:
df["fertilizer_kg_per_ha"] = pd.to_numeric(df["fertilizer_kg_per_ha"], errors="coerce")
df["fertilizer_kg_per_ha"].dtype

In [ ]:
valid_ranges = {"farmer_age": (15, 90), "farm_size_ha": (0.05, 50), "soil_ph": (3.5, 9.5)}

for col, (low, high) in valid_ranges.items():
    bad = ~df[col].between(low, high) & df[col].notna()
    print(f"{col}: {bad.sum()} impossible values set to missing")
    df.loc[~df[col].between(low, high), col] = np.nan

In [ ]:
before = len(df)
df = df.dropna(subset=["yield_kg_per_ha"]).reset_index(drop=True)
print(f"Dropped {before - len(df)} rows with no yield recorded. {len(df)} rows remain.")

> **Why don't we fill in (impute) the other missing values here?**
> If we filled `soil_ph` with the average of the *whole* dataset now, tomorrow's test data would leak into training. Imputation belongs **inside the modelling pipeline**, learned from training data only. We'll do this on Day 2.

## 5. From notebook steps to a reusable function
Copy-pasting cleaning code between notebooks causes errors. The same steps live in **`src/cleaning.py`** as `clean_farm_data()`. Open that file in VS Code and compare it with what you did above.

Then we **test** that the function gives exactly the same result as our step-by-step work:

In [ ]:
from src.cleaning import clean_farm_data, save_processed

df_clean = clean_farm_data(df_raw)
pd.testing.assert_frame_equal(df.reset_index(drop=True), df_clean)
print("✅ Function output matches the step-by-step cleaning")

path = save_processed(df_clean)
print("Saved:", path.relative_to(ROOT))

### 💾 Git checkpoint 2
```bash
git add notebooks/ src/
git commit -m "Clean maize survey data; add cleaning log"
git push
```
Notice that `data/processed/` is listed in `.gitignore`, so it is **not** committed. Anyone can re-create it from the raw data and your code.

## 6. Exploratory data analysis
### 6.1 What does the outcome look like?

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df_clean["yield_kg_per_ha"], bins=30, kde=True, ax=ax)
ax.set(xlabel="Maize yield (kg/ha)", ylabel="Number of farms", title="Distribution of maize yield")
df_clean["yield_kg_per_ha"].describe().round(0)

### 6.2 Yield across LGAs

In [ ]:
order = df_clean.groupby("lga")["yield_kg_per_ha"].median().sort_values().index

fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df_clean, y="lga", x="yield_kg_per_ha", order=order, color="#56B4E9", ax=ax)
ax.set(xlabel="Maize yield (kg/ha)", ylabel="", title="Yield by Local Government Area")
plt.show()

### 6.3 Farm practices

In [ ]:
summary = (df_clean
           .groupby(["improved_seed", "irrigation"])["yield_kg_per_ha"]
           .agg(n="count", mean="mean", median="median", sd="std")
           .round(0))
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
sns.scatterplot(data=df_clean, x="fertilizer_kg_per_ha", y="yield_kg_per_ha", hue="improved_seed", alpha=0.6, ax=axes[0])
axes[0].set(xlabel="Fertiliser (kg/ha)", ylabel="Yield (kg/ha)", title="Fertiliser vs yield")
sns.scatterplot(data=df_clean, x="rainfall_mm", y="yield_kg_per_ha", alpha=0.5, ax=axes[1])
axes[1].set(xlabel="Seasonal rainfall (mm)", title="Rainfall vs yield")
plt.tight_layout()

### 6.4 Correlations between numeric variables

In [ ]:
corr = df_clean.select_dtypes("number").drop(columns="survey_year").corr()

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, ax=ax, annot_kws={"size": 8})
ax.set_title("Correlation matrix")
plt.show()

🔎 **Look closely:** `harvest_sales_naira` is strongly related to yield. Is it a useful *predictor*, or is it *calculated from* yield? Keep this in mind for Day 2.

### 🤖 AI exercise 2: let AI write the code, and you check it
1. Ask: *"Using pandas and seaborn, write code to plot mean maize yield by `pest_incidence` with 95% confidence intervals. Order the categories Absent, Low, Moderate, High. My DataFrame is `df_clean`."*
2. Paste the code below and run it.
3. Ask the AI: *"Explain this code line by line"*. Do you understand **every** line?
4. Check the result: does the chart agree with `df_clean.groupby("pest_incidence")["yield_kg_per_ha"].mean()`?

If you use VS Code with **GitHub Copilot**, try typing a comment such as `# plot mean yield by pest incidence` and pressing Enter.

In [ ]:
# 🤖 Paste and test the AI-generated code here

## ✍️ Exercises
1. Which LGA has the **highest median yield**? Is the difference large compared with the spread *within* LGAs?
2. Do cooperative members get more `extension_visits`? Show it with a table **and** a chart.
3. Save your best chart to `figures/` using `save_figure(fig, "my_chart_name")`.
4. **Bonus:** what percentage of farms use *both* improved seed and irrigation?

In [ ]:
# 1.

In [ ]:
# 2.

In [ ]:
# 3.

## Summary and homework
Today you set up a reproducible project, audited and cleaned data, packaged the cleaning as a tested function, explored the data, and used AI with checking.

### 💾 Final commit for Day 1
```bash
git add .
git commit -m "Day 1: data audit, cleaning and EDA"
git push
```
Then open your repository on github.com and check that your work is there.

**For tomorrow:** if you have your own research dataset (non-confidential), bring it as a CSV or Excel file. You will use it in the mini-project.